<a href="https://colab.research.google.com/github/nehadid/AAI2025/blob/2026fall/ML/house_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
# Generate sample data
from sklearn.datasets import fetch_openml

# Data source: Ames Housing Dataset from OpenML
housing = fetch_openml(name="house_prices", as_frame=True)

df = housing.frame[["GrLivArea", "Neighborhood", "SalePrice"]].copy()

df = df.rename(columns={
    "GrLivArea": "square_footage",
    "Neighborhood": "location",
    "SalePrice": "price"
})

df = df.dropna()

print("Dataset size:", df.shape)
df.head()
# Features and target
X = df[["square_footage", "location"]]
y = df["price"]
# Preprocessing: One-hot encode the location column
preprocessor = ColumnTransformer(
    transformers=[
        ('location', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ['location'])
    ], remainder='passthrough')
# Create pipeline with preprocessing and model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42)
# Train model
model.fit(X_train, y_train)
# Make prediction for a new house: 2000 sq ft in Downtown
new_house = pd.DataFrame({
    "square_footage": [2000],
    "location": ["NAmes"]
})
predicted_price = model.predict(new_house)
print(f"Predicted price for a 2000 sq ft house in Downtown: ${predicted_price[0]:,.2f}")
# Display model coefficients
# Note: If 'Downtown' was an unknown category, its coefficient will be effectively zero due to handle_unknown='ignore'.
# The feature_names extraction needs to be adjusted to reflect all possible categories, including those not seen in training
# if handle_unknown='ignore' is used and we want to see coefficients for all categories present in new data.
# For now, we'll keep the original feature_names extraction, which will only show trained categories + square_footage.
feature_names = (model.named_steps['preprocessor'] \
                 .named_transformers_['location'] \
                 .get_feature_names_out(['location'])).tolist() + \
                ['square_footage']
coefficients = model.named_steps['regressor'].coef_
print("\nModel Coefficients:")
for feature, coef in zip(feature_names, coefficients):
    print(f"{feature}: {coef:.2f}")

print(f"Predicted price for a 2000 sq ft house in NAmes: ${predicted_price[0]:,.2f}")

Dataset size: (1460, 3)
Predicted price for a 2000 sq ft house in Downtown: $199,439.21

Model Coefficients:
location_Blmngtn: 15916.57
location_Blueste: -36576.67
location_BrDale: -51780.44
location_BrkSide: -34350.03
location_ClearCr: 16301.31
location_CollgCr: 19492.68
location_Crawfor: 7062.28
location_Edwards: -40751.05
location_Gilbert: 2269.27
location_IDOTRR: -49248.61
location_MeadowV: -52983.05
location_Mitchel: -9646.37
location_NAmes: -20084.98
location_NPkVill: -17811.23
location_NWAmes: -9598.79
location_NoRidge: 70532.32
location_NridgHt: 97818.13
location_OldTown: -53131.10
location_SWISU: -62843.19
location_Sawyer: -22004.41
location_SawyerW: -4096.70
location_Somerst: 38318.51
location_StoneBr: 96251.05
location_Timber: 37058.39
location_Veenker: 63886.12
square_footage: 76.46
Predicted price for a 2000 sq ft house in NAmes: $199,439.21
